# Sales Forecasting & Demand Planning

## 📈 Business Context

Accurate sales forecasting is critical for optimizing inventory, managing cash flow, and planning resource allocation. This analysis compares multiple time series forecasting models to predict future sales, identifying trends and seasonal patterns to support data-driven decision-making.

## 📊 Objectives

1. Analyze historical sales data for trends and seasonality
2. Test for stationarity (ADF Test)
3. Build and compare forecasting models (SARIMA, Holt-Winters, Prophet)
4. Evaluate model performance (MAE, RMSE, MAPE)
5. Generate a 30-day sales forecast with confidence intervals

## 🔧 Methodology

- **Data**: Synthetic daily sales data (3 years)
- **Techniques**: Seasonal Decomposition, ARIMA/SARIMA, Exponential Smoothing
- **Metrics**: Mean Absolute Error (MAE), Root Mean Squared Error (RMSE)

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
%matplotlib inline

print('✓ Libraries loaded successfully')

## 1. Data Generation

Simulating 3 years of daily sales data with trend and seasonality.

In [ ]:
def generate_sales_data(years=3):
    np.random.seed(42)
    dates = pd.date_range(start='2021-01-01', periods=365*years, freq='D')
    n = len(dates)
    t = np.arange(n)
    
    # Components
    trend = 0.5 * t
    yearly_seasonality = 100 * np.sin(2 * np.pi * t / 365)
    weekly_seasonality = 30 * np.sin(2 * np.pi * t / 7)
    noise = np.random.normal(0, 20, n)
    
    # Holiday Spikes (e.g., Black Friday, Christmas)
    events = np.zeros(n)
    for i, date in enumerate(dates):
        if date.month == 11 and date.day > 20: # Black Friday period
            events[i] = 150
        elif date.month == 12 and date.day > 15: # Christmas rush
            events[i] = 100
            
    sales = 500 + trend + yearly_seasonality + weekly_seasonality + events + noise
    sales = np.maximum(sales, 0)
    
    return pd.DataFrame({'Date': dates, 'Sales': sales}).set_index('Date')

df = generate_sales_data()
print(f"Dataset Shape: {df.shape}")
df.plot(figsize=(14, 6), title='Daily Sales History')
plt.ylabel('Sales ($)')
plt.show()

## 2. Decomposition & Stationarity

Understanding the underlying patterns.

In [ ]:
# Decomposition
decomposition = seasonal_decompose(df['Sales'], model='additive', period=365)
fig = decomposition.plot()
fig.set_size_inches(14, 10)
plt.show()

# ADF Test
result = adfuller(df['Sales'])
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
if result[1] < 0.05:
    print("Result: Stationary")
else:
    print("Result: Non-Stationary (Differencing needed)")

## 3. Model Training

Comparing Holt-Winters and SARIMA.

In [ ]:
# Split Data
train = df.iloc[:-90]
test = df.iloc[-90:]

# Holt-Winters
hw_model = ExponentialSmoothing(
    train['Sales'], 
    seasonal_periods=7, 
    trend='add', 
    seasonal='add'
).fit()
hw_pred = hw_model.forecast(len(test))

# SARIMA (Simplified parameters for speed)
sarima_model = SARIMAX(
    train['Sales'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7)
).fit(disp=False)
sarima_pred = sarima_model.get_forecast(steps=len(test)).predicted_mean

# Evaluation
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")
    return mape

print("Model Evaluation:")
hw_mape = evaluate(test['Sales'], hw_pred, "Holt-Winters")
sarima_mape = evaluate(test['Sales'], sarima_pred, "SARIMA")

## 4. Forecast Visualization

Comparing predictions against actuals.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test.index, test['Sales'], label='Actual', color='black', alpha=0.6)
plt.plot(test.index, hw_pred, label='Holt-Winters', linestyle='--')
plt.plot(test.index, sarima_pred, label='SARIMA', linestyle=':')
plt.title('Forecast Comparison (Test Set)')
plt.legend()
plt.savefig('outputs/forecast_comparison.png')
plt.show()

## 5. Future Forecast

Generating a 30-day outlook.

In [ ]:
future_days = 30
future_dates = pd.date_range(start=test.index[-1] + pd.Timedelta(days=1), periods=future_days)

# Using Holt-Winters as it's often robust for this type of data
future_pred = hw_model.forecast(len(test) + future_days)[-future_days:]

plt.figure(figsize=(14, 6))
plt.plot(df.index[-180:], df['Sales'].iloc[-180:], label='Historical (Last 6 Months)')
plt.plot(future_dates, future_pred, label='30-Day Forecast', color='green', linewidth=2)
plt.title('Sales Forecast: Next 30 Days')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('outputs/future_forecast.png')
plt.show()

## 6. Conclusion

Summary of findings.

In [ ]:
print("="*60)
print("FORECAST SUMMARY")
print("="*60)
print(f"1. Best Model MAPE: {min(hw_mape, sarima_mape):.2f}%")
print(f"2. Trend: Strong positive trend identified.")
print(f"3. Seasonality: Significant weekly and annual patterns.")
print(f"4. Next Month Outlook: Sales expected to continue trend with weekly fluctuations.")